# Extended BKT Model: Hint and Behavior Enhancement

## Overview

This notebook extends the baseline BKT model to create an autism-aware personalized learning system. We add two critical parameters that account for learning support mechanisms particularly relevant for learners with ASD:

### Extension Parameters

Building on the baseline 4-parameter BKT model, we add:

1. **Forgetting Parameter (P(F))**: Models knowledge decay between practice sessions
   - Particularly important for ASD learners who may show variable retention patterns
   - Enables adaptive spacing of practice opportunities

2. **Hint Utilization Factor (α_h)**: Modulates learning rate based on scaffolding support
   - Captures how hints/prompts accelerate skill acquisition
   - Critical for guided instruction approaches common in autism interventions

3. **Behavior Engagement Factor (β_b)**: Adjusts learning/forgetting based on engagement
   - Accounts for attention, persistence, and task engagement
   - Reflects real-world variability in focus and motivation

### Scientific Foundation

- **Forgetting-Aware BKT**: Lee et al. (2023) demonstrated improved prediction accuracy with forgetting parameters
- **Autism-Specific Learning Dynamics**: Research shows ASD learners benefit from explicit modeling of support mechanisms
- **Hint-Aware Knowledge Tracing**: Scaffolded assistance measurably affects learning trajectories

### Objectives

1. Load baseline BKT model from Notebook 03
2. Extend model with forgetting parameter
3. Implement custom update logic for hint and behavior factors
4. Tune hyperparameters (α_h, β_b) using validation data
5. Evaluate extended model performance
6. Compare with baseline to quantify improvement
7. Save optimized model for deployment

---

## 1. Environment Setup

In [ ]:
# ================================================
# Import Required Libraries
# ================================================

# Core data manipulation
import pandas as pd
import numpy as np
from pathlib import Path
from importlib import metadata

# Shared helpers
from src.bkt_engine import PARAM_SCHEMA
from src.split_utils import student_level_split

# BKT baseline model
from pyBKT.models import Model

# Data splitting and evaluation
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    mean_squared_error,
    confusion_matrix
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model persistence
import pickle
import json
from datetime import datetime
import warnings

# Configure environment
warnings.filterwarnings('ignore')
np.random.seed(42)  # Reproducibility
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)


def _package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return 'not installed'

print('✓ All dependencies loaded')
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print('Library versions:')
print(f"  pyBKT   : {_package_version('pyBKT')}")
print(f"  sklearn : {_package_version('scikit-learn')}")
print(f"  numpy   : {_package_version('numpy')}")
print(f"  pandas  : {_package_version('pandas')}")


In [ ]:
# ================================================
# Configure Project Directories
# ================================================

# Define directory paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
FIGURES_DIR = Path("../figures")

# Create output directories
for directory in [DATA_PROCESSED, MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("✓ Directory structure ready")
print(f"  - Models: {MODELS_DIR}")
print(f"  - Results: {RESULTS_DIR}")
print(f"  - Figures: {FIGURES_DIR}")

## 2. Load Data and Baseline Model

In [ ]:
# ================================================
# Load Baseline BKT Model
# ================================================

# Load the trained baseline model from Notebook 03
baseline_model_path = MODELS_DIR / "baseline_bkt_model.pkl"

if not baseline_model_path.exists():
    raise FileNotFoundError(
        f"Baseline model not found at {baseline_model_path}. "
        "Please run Notebook 03 first to train the baseline model."
    )

with open(baseline_model_path, 'rb') as f:
    baseline_model = pickle.load(f)

# Extract baseline parameters
baseline_params = baseline_model.params()

print("✓ Baseline model loaded successfully")
print(f"  - Model file: {baseline_model_path}")
print(f"  - Skills modeled: {len(baseline_params)}")
print("\nBaseline parameters (first skill):")
first_skill = list(baseline_params.keys())[0]
for param, value in baseline_params[first_skill].items():
    print(f"  - {param}: {value:.4f}")

In [ ]:
# ================================================
# Load Autism Learning Data with Extended Features
# ================================================

# Load the synthetic autism data which includes:
# - Standard BKT fields: student_id, skill_name, correct
# - Extended fields: hint_used, behavior_score
data_path = DATA_RAW / "synthetic_autism_data.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Data file not found: {data_path}")

df_raw = pd.read_csv(data_path)

print("✓ Data loaded successfully")
print(f"  - Total records: {len(df_raw):,}")
print(f"  - Columns: {df_raw.columns.tolist()}")

# Check for required extended features
required_features = ['anon_student_id', 'skill_name', 'correct']
extended_features = ['hint_used', 'behavior_score']

# If extended features don't exist, create them
if 'hint_used' not in df_raw.columns:
    print("\n⚠ Creating synthetic 'hint_used' feature...")
    # Simulate hint usage (more likely when student struggles)
    df_raw['hint_used'] = np.where(
        df_raw['correct'] == 0,
        np.random.binomial(1, 0.4, len(df_raw)),  # 40% hint usage on errors
        np.random.binomial(1, 0.1, len(df_raw))   # 10% hint usage on correct
    )

if 'behavior_score' not in df_raw.columns:
    print("⚠ Creating synthetic 'behavior_score' feature...")
    # Simulate engagement score (0.0 to 1.0, with some correlation to correctness)
    base_behavior = np.random.beta(5, 2, len(df_raw))  # Skewed toward higher engagement
    # Boost behavior score slightly for correct answers
    df_raw['behavior_score'] = np.where(
        df_raw['correct'] == 1,
        np.clip(base_behavior + 0.1, 0, 1),
        base_behavior
    )

print("\nData preview with extended features:")
display(df_raw.head())

# Show feature distributions
print("\nExtended feature statistics:")
print(f"  - Hint usage rate: {df_raw['hint_used'].mean():.2%}")
print(f"  - Mean behavior score: {df_raw['behavior_score'].mean():.3f}")
print(f"  - Behavior score std: {df_raw['behavior_score'].std():.3f}")

## 3. Data Preparation and Splitting

We create train and validation sets at the student level to ensure proper evaluation of generalization.

In [ ]:
# ================================================
# Prepare Data for Extended BKT
# ================================================

# Rename columns to BKT-compatible format while keeping extended features
df_extended = df_raw.rename(columns={
    'anon_student_id': 'user_id',
    'skill_name': 'skill_name'
})

# Select columns needed for extended model
columns_needed = ['user_id', 'skill_name', 'correct', 'hint_used', 'behavior_score']
df_extended = df_extended[columns_needed].copy()

# Ensure correct data types
df_extended['correct'] = df_extended['correct'].astype(int)
df_extended['hint_used'] = df_extended['hint_used'].astype(int)
df_extended['behavior_score'] = df_extended['behavior_score'].astype(float)

print("✓ Data prepared for extended BKT")
print(f"  - Total records: {len(df_extended):,}")
print(f"  - Students: {df_extended['user_id'].nunique()}")
print(f"  - Skills: {df_extended['skill_name'].nunique()}")

In [ ]:
# ================================================
# Student-Level Train-Validation Split
# ================================================

# Configuration
VAL_SIZE = 0.2  # 20% of students for validation (hyperparameter tuning)
RANDOM_SEED = 42

train_df, val_df, train_students, val_students = student_level_split(
    df_extended,
    student_col='user_id',
    test_size=VAL_SIZE,
    seed=RANDOM_SEED,
)

# Create train and validation DataFrames
train_df = train_df.copy()
val_df = val_df.copy()

print('=' * 70)
print('TRAIN-VALIDATION SPLIT')
print('=' * 70)
print()
print(f"Total students: {df_extended['user_id'].nunique()}")
print()
print('Training Set:')
print(f"  - Students: {len(train_students)} ({len(train_students)/df_extended['user_id'].nunique()*100:.1f}%)")
print(f"  - Records: {len(train_df):,}")
print(f"  - Accuracy: {train_df['correct'].mean():.2%}")
print(f"  - Hint usage: {train_df['hint_used'].mean():.2%}")
print(f"  - Mean behavior: {train_df['behavior_score'].mean():.3f}")
print()
print('Validation Set:')
print(f"  - Students: {len(val_students)} ({len(val_students)/df_extended['user_id'].nunique()*100:.1f}%)")
print(f"  - Records: {len(val_df):,}")
print(f"  - Accuracy: {val_df['correct'].mean():.2%}")
print(f"  - Hint usage: {val_df['hint_used'].mean():.2%}")
print(f"  - Mean behavior: {val_df['behavior_score'].mean():.3f}")
print()

# Verify no overlap
overlap = set(train_students).intersection(set(val_students))
if len(overlap) == 0:
    print('✓ No student overlap - proper split achieved')
else:
    print(f'⚠ WARNING: {len(overlap)} students in both sets!')


## 4. Add Forgetting Parameter to Baseline

First, we extend the baseline 4-parameter model with forgetting (P(F)).

In [ ]:
# ================================================
# Train BKT Model with Forgetting
# ================================================

# Initialize BKT model with forgetting enabled
# This adds the 5th parameter: P(F) - probability of forgetting
model_with_forgetting = Model(
    seed=RANDOM_SEED,
    num_fits=5  # Multiple random initializations for robustness
)

print('Training BKT with forgetting parameter...')
print(f"  - Training on {len(train_df):,} records")
print(f"  - Students: {len(train_students)}")
print(f"  - Skills: {train_df['skill_name'].nunique()}")
print()

# Prepare training data in BKT format (user_id, skill_name, correct)
train_bkt_format = train_df[['user_id', 'skill_name', 'correct']].copy()

# Fit model with forgetting
# Note: pyBKT's Model class supports forgetting through the forgets parameter
model_with_forgetting.fit(
    data=train_bkt_format,
    forgets=True  # Enable forgetting parameter
)

print('✓ Model with forgetting trained successfully')

# Extract learned parameters and convert them to the canonical schema
raw_forgetting_params = model_with_forgetting.params()
forgetting_params = {}
for skill, params in raw_forgetting_params.items():
    forgetting_params[skill] = {
        'skill': skill,
        'p_l0': float(params.get('prior', params.get('pi_0', 0.0))),
        'p_t': float(params.get('learns', 0.0)),
        'p_g': float(params.get('guesses', 0.0)),
        'p_s': float(params.get('slips', 0.0)),
        'p_f': float(params.get('forgets', 0.0)),
    }

# Display parameters for first skill
print()
print('Learned parameters (with forgetting) - first skill:')
first_skill = list(forgetting_params.keys())[0]
for param, value in forgetting_params[first_skill].items():
    print(f"  - {param}: {value:.4f}" if isinstance(value, (int, float, np.floating)) else f"  - {param}: {value}")


In [ ]:
# ================================================
# Analyze Forgetting Parameters Across Skills
# ================================================

# Extract forgetting rates for all skills
forgetting_analysis = []
for skill, params in forgetting_params.items():
    forgetting_analysis.append({
        'skill': skill,
        'p_l0': params.get('p_l0', 0),
        'p_t': params.get('p_t', 0),
        'p_g': params.get('p_g', 0),
        'p_s': params.get('p_s', 0),
        'p_f': params.get('p_f', 0),
    })

forgetting_df = pd.DataFrame(forgetting_analysis)

print('=' * 70)
print('FORGETTING PARAMETER ANALYSIS')
print('=' * 70)
print()
display(forgetting_df)

print()
print('Forgetting Statistics:')
print(f"  - Mean forgetting rate: {forgetting_df['p_f'].mean():.4f}")
print(f"  - Std forgetting rate: {forgetting_df['p_f'].std():.4f}")
print(f"  - Min forgetting rate: {forgetting_df['p_f'].min():.4f}")
print(f"  - Max forgetting rate: {forgetting_df['p_f'].max():.4f}")

# Identify high-forgetting skills (may need more frequent practice)
high_forgetting_threshold = forgetting_df['p_f'].mean() + forgetting_df['p_f'].std()
high_forgetting_skills = forgetting_df[forgetting_df['p_f'] > high_forgetting_threshold]

if len(high_forgetting_skills) > 0:
    print()
    print(f'⚠ Skills with high forgetting (> {high_forgetting_threshold:.4f}):')
    for _, row in high_forgetting_skills.iterrows():
        print(f"  - {row['skill']}: {row['p_f']:.4f}")


## 5. Implement Extended BKT with Hint and Behavior Factors

Now we implement the custom update logic that incorporates hint usage and behavior engagement.

In [ ]:
# ================================================
# Extended BKT Update Function
# ================================================

def extended_bkt_update(prior, correct, p_l0, p_t, p_g, p_s, p_f, alpha_h, beta_b, hint_used, behavior_score):
    """
    Extended BKT knowledge state update with forgetting, hint, and behavior factors.
    """

    # Step 1: Behavior Modulation
    behavior_factor = 1 + (beta_b * (behavior_score - 0.5))  # Centered at 0.5
    effective_pT = p_t * max(0.1, behavior_factor)
    effective_pF = p_f / max(0.5, behavior_factor)

    # Step 2: Hint Modulation
    if hint_used == 1:
        hint_boost = 1 + alpha_h
        effective_pT = effective_pT * hint_boost

    effective_pT = np.clip(effective_pT, 0.0, 1.0)
    effective_pF = np.clip(effective_pF, 0.0, 1.0)

    # Step 3: Bayesian Evidence Update (standard BKT)
    if correct == 1:
        posterior = (prior * (1 - p_s)) / (
            prior * (1 - p_s) + (1 - prior) * p_g
        )
    else:
        posterior = (prior * p_s) / (
            prior * p_s + (1 - prior) * (1 - p_g)
        )

    # Step 4: Learning and Forgetting Transition
    new_prior = (
        posterior * (1 - effective_pF) +
        (1 - posterior) * effective_pT
    )

    return np.clip(new_prior, 0.0, 1.0)


def predict_with_extended_bkt(df, skill_params, alpha_h, beta_b):
    """
    Generate predictions for a dataset using extended BKT.
    """
    predictions = []
    knowledge_states = []
    student_skill_knowledge = {}

    for idx, row in df.iterrows():
        user_id = row['user_id']
        skill = row['skill_name']
        correct = row['correct']
        hint_used = row['hint_used']
        behavior_score = row['behavior_score']

        params = skill_params.get(skill, {})
        pL = params.get('p_l0', 0.5)
        pT = params.get('p_t', 0.15)
        pG = params.get('p_g', 0.25)
        pS = params.get('p_s', 0.20)
        pF = params.get('p_f', 0.10)

        key = (user_id, skill)
        if key not in student_skill_knowledge:
            student_skill_knowledge[key] = pL

        current_knowledge = student_skill_knowledge[key]
        pred_prob = current_knowledge * (1 - pS) + (1 - current_knowledge) * pG
        predictions.append(pred_prob)

        new_knowledge = extended_bkt_update(
            current_knowledge, correct, pL, pT, pG, pS, pF,
            alpha_h, beta_b, hint_used, behavior_score
        )

        student_skill_knowledge[key] = new_knowledge
        knowledge_states.append(new_knowledge)

    return np.array(predictions), np.array(knowledge_states)


print('✓ Extended BKT functions defined')
print('  - extended_bkt_update: Knowledge state update with hint/behavior')
print('  - predict_with_extended_bkt: Batch prediction function')


## 6. Hyperparameter Tuning (α_h and β_b)

We use grid search on the validation set to find optimal values for the hint boost and behavior modulation factors.

In [ ]:
# ================================================
# Grid Search for Optimal Hyperparameters
# ================================================

print("Starting hyperparameter grid search...")
print("This may take several minutes...\n")

# Define search space
# alpha_h: Hint boost factor (how much hints accelerate learning)
alpha_h_candidates = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

# beta_b: Behavior modulation factor (how much engagement affects learning/forgetting)
beta_b_candidates = [0.0, 0.5, 1.0, 1.5, 2.0]

# Track results
grid_results = []

# Sample validation data for faster grid search
# Use enough data to get stable estimates but not so much it's slow
val_sample_size = min(10000, len(val_df))
val_sample = val_df.sample(n=val_sample_size, random_state=42)

print(f"Grid search configuration:")
print(f"  - alpha_h candidates: {alpha_h_candidates}")
print(f"  - beta_b candidates: {beta_b_candidates}")
print(f"  - Total combinations: {len(alpha_h_candidates) * len(beta_b_candidates)}")
print(f"  - Validation sample size: {val_sample_size:,}")
print()

# Iterate through all combinations
best_auc = 0
best_params = (0, 0)
best_metrics = {}

for alpha_h in alpha_h_candidates:
    for beta_b in beta_b_candidates:
        # Generate predictions with current hyperparameters
        preds, _ = predict_with_extended_bkt(
            val_sample,
            forgetting_params,
            alpha_h,
            beta_b
        )
        
        # Calculate metrics
        actuals = val_sample['correct'].values
        
        # Accuracy (using 0.5 threshold)
        pred_binary = (preds >= 0.5).astype(int)
        accuracy = accuracy_score(actuals, pred_binary)
        
        # RMSE
        rmse = np.sqrt(mean_squared_error(actuals, preds))
        
        # AUC (if both classes present)
        try:
            auc = roc_auc_score(actuals, preds)
        except:
            auc = 0.5  # Fallback if only one class
        
        # Store results
        grid_results.append({
            'alpha_h': alpha_h,
            'beta_b': beta_b,
            'accuracy': accuracy,
            'rmse': rmse,
            'auc': auc
        })
        
        # Track best configuration (using AUC as primary metric)
        if auc > best_auc:
            best_auc = auc
            best_params = (alpha_h, beta_b)
            best_metrics = {
                'accuracy': accuracy,
                'rmse': rmse,
                'auc': auc
            }
        
        # Print progress every 5 iterations
        if len(grid_results) % 5 == 0:
            print(f"Tested {len(grid_results)}/{len(alpha_h_candidates) * len(beta_b_candidates)} combinations...")

# Convert results to DataFrame
grid_df = pd.DataFrame(grid_results)

print("\n" + "=" * 70)
print("GRID SEARCH RESULTS")
print("=" * 70)
print(f"\nBest Parameters:")
print(f"  - alpha_h (hint boost): {best_params[0]:.2f}")
print(f"  - beta_b (behavior modulation): {best_params[1]:.2f}")
print(f"\nBest Performance:")
print(f"  - Accuracy: {best_metrics['accuracy']:.4f}")
print(f"  - RMSE: {best_metrics['rmse']:.4f}")
print(f"  - AUC: {best_metrics['auc']:.4f}")

print("\nTop 5 Configurations:")
top_configs = grid_df.nlargest(5, 'auc')
display(top_configs)

In [ ]:
# ================================================
# Visualize Grid Search Results
# ================================================

# Create heatmap of AUC scores across hyperparameter combinations
pivot_table = grid_df.pivot(index='beta_b', columns='alpha_h', values='auc')

plt.figure(figsize=(12, 6))
sns.heatmap(
    pivot_table,
    annot=True,
    fmt='.4f',
    cmap='YlOrRd',
    cbar_kws={'label': 'AUC Score'}
)
plt.title('Hyperparameter Grid Search: AUC Performance', fontsize=14, fontweight='bold')
plt.xlabel('alpha_h (Hint Boost Factor)', fontsize=12)
plt.ylabel('beta_b (Behavior Modulation Factor)', fontsize=12)
plt.tight_layout()

# Save figure
grid_fig_path = FIGURES_DIR / "hyperparameter_grid_search.png"
plt.savefig(grid_fig_path, dpi=300, bbox_inches='tight')
print(f"✓ Grid search visualization saved to: {grid_fig_path}")

plt.show()